In [37]:
%reset -f

In [38]:
# merge the two tables
import pandas as pd
train_transaction = pd.read_csv('rawdata/train_transaction.csv')
train_identity = pd.read_csv('rawdata/train_identity.csv')
train_combined = train_transaction.merge(train_identity, on='TransactionID', how='left')
train_combined = train_combined.copy() # clear up the 'fragmentation' warning

In [45]:
print(train_transaction.info())

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), str(14)
memory usage: 1.7 GB
None


In [48]:
print(train_transaction.columns)

Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5',
       ...
       'V330', 'V331', 'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338',
       'V339'],
      dtype='str', length=394)


In [46]:
print(train_identity.info())

<class 'pandas.DataFrame'>
RangeIndex: 144233 entries, 0 to 144232
Data columns (total 41 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionID  144233 non-null  int64  
 1   id_01          144233 non-null  float64
 2   id_02          140872 non-null  float64
 3   id_03          66324 non-null   float64
 4   id_04          66324 non-null   float64
 5   id_05          136865 non-null  float64
 6   id_06          136865 non-null  float64
 7   id_07          5155 non-null    float64
 8   id_08          5155 non-null    float64
 9   id_09          74926 non-null   float64
 10  id_10          74926 non-null   float64
 11  id_11          140978 non-null  float64
 12  id_12          144233 non-null  str    
 13  id_13          127320 non-null  float64
 14  id_14          80044 non-null   float64
 15  id_15          140985 non-null  str    
 16  id_16          129340 non-null  str    
 17  id_17          139369 non-null  float64


In [47]:
print(train_identity.columns)

Index(['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06',
       'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14',
       'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22',
       'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30',
       'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType', 'DeviceInfo'],
      dtype='str')


In [49]:
# convert TransactionDT to a day number
## TransactionDT is in seconds - divide by 86400 to get days
train_combined['TransactionDay'] = (train_combined['TransactionDT'] / 86400).astype(int)
## inspect
print(train_combined['TransactionDay'].describe())

count    590540.000000
mean         84.729199
std          53.437277
min           1.000000
25%          35.000000
50%          84.000000
75%         130.000000
max         182.000000
Name: TransactionDay, dtype: float64


In [50]:
# compute D1n (the card's birthday)
## D1n = TransactionDay - D1
train_combined['D1n'] = train_combined['TransactionDay'] - train_combined['D1']
## inspect - does this look stable for the same card
print(train_combined[['card1', 'D1', 'TransactionDay', 'D1n']].head(20))

    card1     D1  TransactionDay    D1n
0   13926   14.0               1  -13.0
1    2755    0.0               1    1.0
2    4663    0.0               1    1.0
3   18132  112.0               1 -111.0
4    4497    0.0               1    1.0
5    5937    0.0               1    1.0
6   12308    0.0               1    1.0
7   12695    0.0               1    1.0
8    2803    0.0               1    1.0
9   17399   61.0               1  -60.0
10  16496    1.0               1    0.0
11   4461    0.0               1    1.0
12   3786   72.0               1  -71.0
13  12866   46.0               1  -45.0
14  11839    0.0               1    1.0
15   7055    0.0               1    1.0
16   1790    0.0               1    1.0
17  11492    0.0               1    1.0
18   4663    0.0               1    1.0
19   7005   62.0               1  -61.0


In [41]:
# build the UID
uid_column = (
    train_combined['card1'].astype(str) + '_' +
    train_combined['addr1'].astype(str) + '_' +
    train_combined['D1n'].astype(str)
)
# concatenate uid_column to the train_combined
train_combined = pd.concat([train_combined, uid_column.rename('UID')], axis=1)
print(train_combined['UID'].unique()) # how many unique "cards" did we find
print(train_combined['UID'].value_counts()[:10]) # most frequent UIDs

<StringArray>
[ '13926_315.0_-13.0',     '2755_325.0_1.0',     '4663_330.0_1.0',
 '18132_476.0_-111.0',     '4497_420.0_1.0',     '5937_272.0_1.0',
    '12308_126.0_1.0',    '12695_325.0_1.0',     '2803_337.0_1.0',
  '17399_204.0_-60.0',
 ...
  '15813_441.0_182.0',  '8431_325.0_-443.0',  '10640_330.0_182.0',
  '16873_110.0_182.0',   '2198_315.0_182.0',   '7826_387.0_182.0',
 '11942_310.0_-451.0', '12037_231.0_-133.0',  '10444_204.0_182.0',
  '12037_231.0_182.0']
Length: 199071, dtype: str
UID
15775_330.0_129.0     1414
9500_126.0_-85.0       446
8900_231.0_-60.0       232
8528_387.0_-159.0      215
12741_143.0_-202.0     196
7207_204.0_-465.0      191
13597_191.0_-48.0      148
4121_476.0_-8.0        142
12695_325.0_-342.0     123
2884_204.0_-32.0       118
Name: count, dtype: int64


In [66]:
# sanity check on the UID
## do fraudulent transactions cluster in specific UIDs?
grouped_uid_isfraud = train_combined.groupby('UID')['isFraud'].agg(['sum', 'count', 'mean'])
grouped_uid_isfraud.columns = ['fraud_count', 'total_txns', 'fraud_rate']
grouped_uid_isfraud = grouped_uid_isfraud.sort_values('fraud_rate', ascending=False)
print(grouped_uid_isfraud.head(20))
# UIDs with more than 1 transaction
# Only look at UIDs with more than 1 transaction (real repeat behavior)
repeat_uids = grouped_uid_isfraud[grouped_uid_isfraud['total_txns'] > 1]
print('--------------')
print(repeat_uids['fraud_rate'].describe())
print('--------------')
print((repeat_uids['fraud_rate'].isin([0, 1])).mean())  # % of repeat UIDs that are "pure"

                    fraud_count  total_txns  fraud_rate
UID                                                    
16659_264.0_5.0               3           3         1.0
13844_472.0_75.0              1           1         1.0
18097_126.0_107.0             1           1         1.0
15066_123.0_2.0               1           1         1.0
6957_310.0_44.0               1           1         1.0
10289_325.0_67.0              1           1         1.0
6957_441.0_173.0              3           3         1.0
6958_472.0_167.0              1           1         1.0
16346_284.0_138.0             1           1         1.0
13844_476.0_57.0              1           1         1.0
16075_299.0_167.0             1           1         1.0
14869_337.0_136.0             1           1         1.0
12695_126.0_-184.0           15          15         1.0
1675_110.0_123.0              1           1         1.0
14098_325.0_111.0             3           3         1.0
15863_428.0_28.0              1           1     